1. **Dynamic Partition Pruning (DPP) in PySpark**:  
   Dynamic Partition Pruning is an optimization technique in Spark SQL that reduces the amount of data read during join operations. When joining a large fact table with a smaller dimension table on a partitioned column, DPP ensures that only relevant partitions from the fact table (those matching the join keys from the dimension table) are scanned, improving query performance and reducing resource usage.


In [0]:
2. Write a pyspark code to read a file, filter data, add a new column and write the data in a delta table.

# Step 1: Ingest data efficiently using Spark's parallelism
raw_df = spark.read.format("parquet").option("mergeSchema", "false").load("/mnt/data/landing/")

# Step 2: Apply schema-on-read and validate data quality
validated_df = raw_df.filter("required_column IS NOT NULL")

# Step 3: Partition data by relevant columns (e.g., date, region) for optimized storage and access
partitioned_df = validated_df.withColumn("partition_date", validated_df["event_date"])

# Step 4: Write data to Delta Lake for ACID transactions and scalable storage
partitioned_df.write.format("delta").mode("append").partitionBy("partition_date").save("/mnt/data/delta/events/")



In [0]:
# 3. Cache vs persist
# result_df.cache() stores DataFrame in memory for fast access

# Alternatively, use persist() to specify storage level
from pyspark.storagelevel import StorageLevel

# Persist with MEMORY_AND_DISK for fault tolerance
result_df.persist(StorageLevel.MEMORY_AND_DISK)

In [0]:
4. **Building a Data Factory Pipeline to Extract 10 TB Data Daily from a SAP Table**

   - **Step 1: Create Linked Services**
     - Set up a Linked Service for SAP (e.g., SAP Table, SAP BW, or SAP HANA connector).
     - Set up a Linked Service for your destination (e.g., Azure Data Lake Storage, Blob Storage, or Delta Lake).
     - Set up an Integration Runtime (IR) connection for secure and scalable data movement (e.g., Azure, Self-hosted, or Managed IR).

   - **Step 2: Define Datasets**
     - Create a source dataset for the SAP table.
     - Create a sink dataset for the destination storage.

   - **Step 3: Design the Pipeline**
     - Use the Copy Data activity to extract data from SAP and load into your destination.
     - Enable parallelism and partitioning in the Copy Data activity for high throughput (e.g., partition by date, range, or custom column).
     - Use Data Flows for any required transformations.

   - **Step 4: Optimize for Large Data Volumes**
     - Use SAP ODP or CDC (Change Data Capture) for incremental loads if supported.
     - Tune batch size, degree of parallelism, and integration runtime resources.
     - Monitor pipeline performance and handle retries/failures.

   - **Step 5: Schedule and Monitor**
     - Schedule the pipeline to run daily using triggers.
     - Monitor pipeline runs and set up alerts for failures.

   - **References:**
     - [Azure Data Factory SAP connectors documentation](https://learn.microsoft.com/en-us/azure/data-factory/connector-sap-table)
     - [Best practices for large-scale data movement](https://learn.microsoft.com/en-us/azure/data-factory/copy-activity-performance)

In [0]:
5. How to build an incremental load pipeline.

# Step 1: Read latest processed watermark (e.g., max event_date) from Delta Lake
watermark_df = spark.sql("SELECT MAX(event_date) AS last_processed FROM delta.`/mnt/data/delta/events/`")
last_processed = watermark_df.collect()[0]['last_processed']

# Step 2: Ingest only new data from landing zone based on watermark
incremental_df = spark.read.format("parquet").option("mergeSchema", "false").load("/mnt/data/landing/") \
    .filter(f"event_date > '{last_processed}'")

# Step 3: Validate and partition incremental data
validated_incremental_df = incremental_df.filter("required_column IS NOT NULL") \
    .withColumn("partition_date", incremental_df["event_date"])

# Step 4: Append incremental data to Delta Lake
validated_incremental_df.write.format("delta").mode("append").partitionBy("partition_date").save("/mnt/data/delta/events/")

# Step 5: Update watermark tracking table if needed
spark.sql(f"""
MERGE INTO delta.`/mnt/data/delta/watermark_tracking/` AS t
USING (SELECT '{last_processed}' AS last_processed) AS s
ON t.id = 1
WHEN MATCHED THEN UPDATE SET t.last_processed = s.last_processed
WHEN NOT MATCHED THEN INSERT (id, last_processed) VALUES (1, s.last_processed)
""")

In [0]:
6. groupBy vs reduceBy in Spark.
  

# groupBy: Groups data by key, then applies aggregation. May cause large shuffles as all values for a key are sent to a single node.
# reduceBy: Combines values for each key locally before shuffling, reducing data transfer and improving performance.

# Example:
rdd = spark.sparkContext.parallelize([("a", 1), ("b", 2), ("a", 3)])

# groupBy example (returns grouped values)
grouped = rdd.groupBy(lambda x: x[0])
# Aggregation after grouping
grouped_sum = grouped.mapValues(lambda vals: sum([v[1] for v in vals]))

# reduceBy example (reduces values by key during shuffle)
reduced = rdd.reduceByKey(lambda x, y: x + y)

display(grouped_sum)
display(reduced)